# 原始 GPT-2 Baseline 架构解析

源码导航：[`GPT2LMHeadModel`](../../../core/model/gpt2.py#L67)、[`wte`](../../../core/model/gpt2.py#L77)、[`wpe`](../../../core/model/gpt2.py#L78)。

本笔记本旨在拆解 GPT-2 的整体架构，并演示如何将其微缩化以便在本地调试。通过阅读本篇，你将理解 GPT-2 的网络流向以及它与现代 Transformer 的细微区别。

## 1. 结构全景

GPT-2 是一个 **Decoder-only** 的架构，其数据流向如下：

```text
Input Tokens (B, T)
    │
    ▼
[ Embedding 层 ] ──► Token Embedding (wte) + Positional Embedding (wpe)
    │
    ▼
[ Transformer Blocks × N ] ──► LayerNorm -> Dropout -> MHA -> Residual -> LayerNorm -> MLP -> Residual
    │
    ▼
[ 输出层 ] ──► Final LayerNorm (ln_f) -> Language Model Head (Linear)
```

### 关键特性：Pre-LN 残差结构
与原始 Transformer (Post-LN) 不同，GPT-2 使用 Pre-LN，这使得训练更加稳定。

$$x_{next} = x + \text{SubLayer}(\text{LayerNorm}(x))$$

---

## 2. 核心组件拆解

### 2.1 嵌入层 (Embedding)
GPT-2 使用**学习得到**的绝对位置编码（Learned Absolute PE），而不是正弦余弦编码。

源码对应：`GPT2LMHeadModel` 中的 [`wte`](../../../core/model/gpt2.py#L77) 和 [`wpe`](../../../core/model/gpt2.py#L78)

In [ ]:
import torch
import torch.nn as nn

vocab_size = 50257
n_embd = 768
block_size = 1024 # 最大序列长度

wte = nn.Embedding(vocab_size, n_embd) # Token Embedding
wpe = nn.Embedding(block_size, n_embd) # Positional Embedding

# 模拟输入
idx = torch.tensor([[1, 2, 3, 4]]) # (B, T)
T = idx.size(1)
pos = torch.arange(0, T, dtype=torch.long) # (T)

tok_emb = wte(idx) # (B, T, n_embd)
pos_emb = wpe(pos) # (T, n_embd)

x = tok_emb + pos_emb # 广播相加
print(f"输入 Embedding 形状: {x.shape}")

### 2.2 权重共享 (Weight Tying)
GPT-2 为了节省参数，让输入 Embedding (`wte`) 和最后的输出映射层 (`lm_head`) **共享相同的权重矩阵**。

源码对应：[`GPT2LMHeadModel.__init__`](../../../core/model/gpt2.py#L85) 中的 `self.lm_head.weight = self.wte.weight`

In [ ]:
lm_head = nn.Linear(n_embd, vocab_size, bias=False)
# 执行共享
lm_head.weight = wte.weight 

print(f"WTE 权重 ID: {id(wte.weight)}")
print(f"LM Head 权重 ID: {id(lm_head.weight)}")
assert id(wte.weight) == id(lm_head.weight)

---

## 3. 与 HuggingFace 对齐

本项目支持直接从 HuggingFace 加载权重。由于 HF 使用的是 `Conv1D`（权重形状为 `[in, out]` 且内置转置），而 PyTorch 标准 `Linear` 的权重形状为 `[out, in]`，在加载时需要进行一次转置。

源码对应：[`GPT2LMHeadModel.from_pretrained_hf`](../../../core/model/gpt2.py#L184)

---

## 4. 不同尺寸概览

| 配置 | 层数 (L) | 头数 (H) | 维度 (D) | 参数量 |
| --- | --- | --- | --- | --- |
| `gpt2_tiny` | 2 | 2 | 64 | ~1M |
| `gpt2_124m` | 12 | 12 | 768 | 124M |
| `gpt2_1558m` | 48 | 25 | 1600 | 1.5B |

---

## 5. 延伸阅读与参考资料

### 核心论文 (Paper)
- **GPT-2 技术报告**: Radford et al., 2019. *Language Models are Unsupervised Multitask Learners*.
- **Transformer 奠基**: Vaswani et al., 2017. *Attention Is All You Need*.

### 优质资源 (Blog/Code)
- **Karpathy nanoGPT**: [GitHub](https://github.com/karpathy/nanoGPT) (本项目的重要参考)
- **Jay Alammar**: *The Illustrated GPT-2*. [Link](https://jalammar.github.io/illustrated-gpt2/)

---
> [03 · Attention](../attention/03_attention_mha.ipynb)